In [ ]:
# -*- coding: utf-8 -*-
"""
Traffic Event Intelligence System
Modular approach implementing 5 distinct models for traffic disruption management.
"""
!pip install faiss-cpu # Install faiss
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN
from sklearn.metrics import mean_absolute_error, classification_report
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings('ignore')

# =========================================================
# 0. Data Processing Module
# =========================================================
class DataProcessor:
    def __init__(self, url):
        self.url = url
        self.df = None
        self.features_base = [
            'event_cause', 'zone', 'corridor', 'junction',
            'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'crowd_event_flag'
        ]

    def load_and_preprocess(self):
        print("Loading and cleaning data...")
        self.df = pd.read_csv(self.url)

        # Datetime Processing
        self.df['start_datetime'] = pd.to_datetime(self.df['start_datetime'], errors='coerce')
        self.df['closed_datetime'] = pd.to_datetime(self.df['closed_datetime'], errors='coerce')
        self.df['resolution_time'] = (self.df['closed_datetime'] - self.df['start_datetime']).dt.total_seconds() / 60

        # Clean unrealistic durations (0 < mins < 24 hrs)
        self.df = self.df[(self.df['resolution_time'] > 0) & (self.df['resolution_time'] < 1440)].copy()

        # Time and Categorical Features
        self.df['hour_of_day'] = self.df['start_datetime'].dt.hour
        self.df['day_of_week'] = self.df['start_datetime'].dt.dayofweek
        self.df['month'] = self.df['start_datetime'].dt.month
        self.df['is_weekend'] = (self.df['day_of_week'] >= 5).astype(int)

        self.df['description'] = self.df['description'].fillna('').str.lower()
        crowd_keywords = ['procession', 'protest', 'vip', 'minister', 'festival', 'march', 'public']
        self.df['crowd_event_flag'] = self.df['description'].apply(lambda x: int(any(k in x for k in crowd_keywords)))

        self.df['requires_road_closure_num'] = self.df['requires_road_closure'].astype(int)

        # Impact Classification Target
        self.df['impact_class'] = pd.cut(
            self.df['resolution_time'],
            bins=[0, 30, 120, np.inf],
            labels=['Low', 'Medium', 'High']
        )

        # Impute categorical NAs
        for col in ['event_cause', 'zone', 'corridor', 'junction']:
            self.df[col] = self.df[col].fillna('Unknown').astype('category')

        return self.df

    def get_train_test_split(self):
        df_model = self.df.dropna(subset=['resolution_time', 'impact_class'] + self.features_base).copy()
        train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42)

        # Compute historical risk (Leak-Free: calculated on train, applied to train/test)
        junction_stats = train_df.groupby('junction', observed=False).agg(
            hist_avg_resolution_time=('resolution_time', 'mean'),
            hist_event_count=('id', 'count'),
            hist_road_closure_rate=('requires_road_closure_num', 'mean')
        ).reset_index()

        train_df = train_df.merge(junction_stats, on='junction', how='left')
        test_df = test_df.merge(junction_stats, on='junction', how='left')

        global_avg = train_df['resolution_time'].mean()
        test_df['hist_avg_resolution_time'] = test_df['hist_avg_resolution_time'].fillna(global_avg)
        test_df['hist_event_count'] = test_df['hist_event_count'].fillna(1)
        test_df['hist_road_closure_rate'] = test_df['hist_road_closure_rate'].fillna(0)

        features = self.features_base + ['hist_avg_resolution_time', 'hist_event_count', 'hist_road_closure_rate']
        return train_df, test_df, features


# =========================================================
# Model 1: Duration Prediction Model (LightGBM Regressor)
# =========================================================
class DurationPredictor:
    def __init__(self):
        self.model = lgb.LGBMRegressor(objective='regression', n_estimators=300, learning_rate=0.05, random_state=42)

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        print("Model 1 (Duration Predictor) trained successfully.")

    def predict(self, X):
        return self.model.predict(X)


# =========================================================
# Model 2: Impact Classification Model (LightGBM Classifier)
# =========================================================
class ImpactClassifier:
    def __init__(self):
        self.model = lgb.LGBMClassifier(objective='multiclass', n_estimators=300, random_state=42)

    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        print("Model 2 (Impact Classifier) trained successfully.")

    def predict(self, X):
        return self.model.predict(X)


# =========================================================
# Model 3: Geospatial Hotspot Detection (DBSCAN)
# =========================================================
class HotspotDetector:
    def __init__(self, eps_km=0.5, min_samples=5):
        self.eps_radians = eps_km / 6371.0088  # Convert km to radians for haversine
        self.dbscan = DBSCAN(eps=self.eps_radians, min_samples=min_samples, metric='haversine', algorithm='ball_tree')
        self.hotspots = None

    def fit_and_identify(self, df):
        coords = df[['latitude', 'longitude']].dropna()
        clusters = self.dbscan.fit_predict(np.radians(coords))
        df.loc[coords.index, 'cluster'] = clusters

        self.hotspots = (
            df[df['cluster'] != -1]
            .groupby('cluster')
            .agg(
                avg_duration=('resolution_time', 'mean'),
                event_count=('id', 'count'),
                center_lat=('latitude', 'mean'),
                center_lon=('longitude', 'mean')
            )
        )
        print("Model 3 (Hotspot Detector) processed spatial clusters.")

    def check_if_hotspot(self, lat, lon):
        # Naive distance check to nearest hotspot center
        if self.hotspots is None or pd.isna(lat) or pd.isna(lon):
            return False, None

        for index, row in self.hotspots.iterrows():
            dist = np.sqrt((row['center_lat'] - lat)**2 + (row['center_lon'] - lon)**2)
            if dist < 0.01:  # rough proxy for overlapping with a cluster region
                return True, row
        return False, None


# =========================================================
# Model 4: Semantic Event Retrieval (FAISS)
# =========================================================
class SemanticEventRetriever:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        self.df_reference = None

    def fit(self, df):
        self.df_reference = df.dropna(subset=['description']).reset_index(drop=True)
        texts = (
            self.df_reference['event_cause'].astype(str) + " | " +
            self.df_reference['junction'].astype(str) + " | " +
            self.df_reference['description'].astype(str)
        ).tolist()

        embeddings = self.model.encode(texts, normalize_embeddings=True).astype('float32')
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)
        print("Model 4 (Semantic Retriever) indexed historical events.")

    def search_similar(self, event_dict, top_k=3):
        query_text = f"{event_dict.get('event_cause','')} | {event_dict.get('junction','')} | {event_dict.get('description','')}"
        query_embed = self.model.encode([query_text], normalize_embeddings=True).astype('float32')

        scores, idxs = self.index.search(query_embed, top_k)
        results = self.df_reference.iloc[idxs[0]][['event_cause', 'junction', 'description', 'impact_class']]
        results['similarity_score'] = scores[0]
        return results


# =========================================================
# Model 5: Resource Recommendation Engine (Rule-Based)
# =========================================================
class ResourceRecommender:
    def __init__(self):
        self.base_allocations = {
            'Low': {'officers': 2, 'barricades': 5, 'diversion_plan': 'Monitor Local Traffic'},
            'Medium': {'officers': 8, 'barricades': 15, 'diversion_plan': 'Immediate Alternate Route Assignment'},
            'High': {'officers': 20, 'barricades': 40, 'diversion_plan': 'City-wide Zone Diversion Plan Alpha'}
        }

    def recommend(self, impact_class, event_cause, road_closure, in_hotspot):
        # Default allocation based on predicted impact
        allocation = self.base_allocations.get(impact_class, self.base_allocations['Medium']).copy()

        # Modifier rules
        if road_closure:
            allocation['officers'] += 5
            allocation['barricades'] += 20

        if in_hotspot:
            allocation['officers'] += 3
            allocation['diversion_plan'] = "Critical: Pre-approved Hotspot Diversion Strategy Deployed"

        if event_cause.lower() in ['procession', 'protest', 'vip_movement']:
            allocation['officers'] += 15
            allocation['barricades'] += 10

        return allocation


# =========================================================
# End-to-End Orchestrator (Simulating the Flowchart)
# =========================================================
def run_pipeline():
    url = "https://raw.githubusercontent.com/WaniAnimesh/FlowState-Ops/refs/heads/main/Astram%20event%20data_anonymized.csv"

    # 1. Setup Phase
    dp = DataProcessor(url)
    df = dp.load_and_preprocess()
    train_df, test_df, feature_cols = dp.get_train_test_split()

    X_train, y_train_reg = train_df[feature_cols], train_df['resolution_time']
    y_train_clf = train_df['impact_class']

    # Initialize Models Separately
    model_1 = DurationPredictor()
    model_1.train(X_train, y_train_reg)

    model_2 = ImpactClassifier()
    model_2.train(X_train, y_train_clf)

    model_3 = HotspotDetector()
    model_3.fit_and_identify(df)

    model_4 = SemanticEventRetriever()
    model_4.fit(df)

    model_5 = ResourceRecommender()

    print("\n" + "="*60)
    print("🚦 OPERATIONAL EVENT FLOW: SIMULATING INCOMING INCIDENT 🚦")
    print("="*60)

    # 2. Simulate "New Event"
    # Mapping new dictionary properties to model-readable Dataframe layout
    new_event = {
        'event_cause': 'procession',
        'zone': 'Central Zone 2',
        'corridor': 'MG Road',
        'junction': 'Trinity Circle',
        'latitude': 12.973,  # Approximate coord
        'longitude': 77.615,
        'description': 'Massive VIP procession causing absolute gridlock on primary roads.',
        'requires_road_closure_num': 1,
        'hour_of_day': 17, # 5 PM (Rush Hour)
        'day_of_week': 4,  # Friday
        'month': 5,
        'is_weekend': 0,
        'crowd_event_flag': 1,
        # Default Historical features just for simulated model format
        'hist_avg_resolution_time': 120,
        'hist_event_count': 5,
        'hist_road_closure_rate': 0.8
    }

    # Format for Models 1 & 2
    new_event_df = pd.DataFrame([new_event])[feature_cols]
    for col in ['event_cause', 'zone', 'corridor', 'junction']:
        new_event_df[col] = new_event_df[col].astype('category')

    print("\n📩 [STEP 1] INCOMING EVENT REGISTERED:")
    print(f"Cause: {new_event['event_cause'].title()} | Location: {new_event['junction']} | Road Closure Req: Yes")

    # Step 2: Prediction Models (1 & 2)
    pred_duration = model_1.predict(new_event_df)[0]
    pred_impact = model_2.predict(new_event_df)[0]
    print(f"\n🔮 [STEP 2 & 3] AI IMPACT PREDICTIONS (Model 1 & 2):")
    print(f" -> Predicted Duration: {pred_duration:.1f} minutes")
    print(f" -> Predicted Class:   [{pred_impact.upper()}] Severity")

    # Step 3: Geographic Context (Model 3)
    is_hotspot, _ = model_3.check_if_hotspot(new_event['latitude'], new_event['longitude'])
    print(f"\n🗺️ [STEP 4] HOTSPOT DETECTION (Model 3):")
    print(f" -> Location resides in a recurring hotspot: {'YES ⚠️' if is_hotspot else 'NO'}")

    # Step 4: Semantic Context (Model 4)
    print("\n🔍 [STEP 5] SIMILAR HISTORICAL EVENTS (Model 4):")
    sim_events = model_4.search_similar(new_event, top_k=2)
    for idx, row in sim_events.iterrows():
        print(f" -> Past '{row['event_cause']}' | Impact: {row['impact_class']} | Similarity: {row['similarity_score']:.2f}")

    # Step 5: Resource Engine (Model 5)
    print("\n🛠️ [STEP 6] RESOURCE RECOMMENDATION ENGINE (Model 5):")
    resources = model_5.recommend(pred_impact, new_event['event_cause'], True, is_hotspot)
    print(f" -> Manpower Suggested: {resources['officers']} Officers")
    print(f" -> Barricades Required: {resources['barricades']} Units")
    print(f" -> Diversion Protocol:  {resources['diversion_plan']}")

    # Final Output Summary
    print("\n" + "*"*60)
    print("✅ OPERATIONAL ACTION PLAN GENERATED AUTOMATICALLY ✅")
    print("*"*60)

if __name__ == "__main__":
    run_pipeline()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.4 MB/s eta 0:00:00
Loading and cleaning data...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000507 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 341
[LightGBM] [Info] Number of data points in the train set: 1968, number of used features: 11
[LightGBM] [Info] Start training from score 98.521974
Model 1 (Duration Predictor) trained successfully.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000193 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 341
[LightGBM] [Info] Number of data points in the train set: 1968, number of used features: 11
[LightGBM] [Info] Start training from score -1.982654
[LightGBM] [Info] Start trainin

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model 4 (Semantic Retriever) indexed historical events.

🚦 OPERATIONAL EVENT FLOW: SIMULATING INCOMING INCIDENT 🚦

📩 [STEP 1] INCOMING EVENT REGISTERED:
Cause: Procession | Location: Trinity Circle | Road Closure Req: Yes

🔮 [STEP 2 & 3] AI IMPACT PREDICTIONS (Model 1 & 2):
 -> Predicted Duration: 97.3 minutes
 -> Predicted Class:   [MEDIUM] Severity

🗺️ [STEP 4] HOTSPOT DETECTION (Model 3):
 -> Location resides in a recurring hotspot: NO

🔍 [STEP 5] SIMILAR HISTORICAL EVENTS (Model 4):
 -> Past 'procession' | Impact: Low | Similarity: 0.64
 -> Past 'procession' | Impact: High | Similarity: 0.56

🛠️ [STEP 6] RESOURCE RECOMMENDATION ENGINE (Model 5):
 -> Manpower Suggested: 28 Officers
 -> Barricades Required: 45 Units
 -> Diversion Protocol:  Immediate Alternate Route Assignment

************************************************************
✅ OPERATIONAL ACTION PLAN GENERATED AUTOMATICALLY ✅
************************************************************
